In [1]:
import numpy as np
import torch
from torch.utils.data import DataLoader
import os
from autoenkoop import *
import scipy

DEVICE = "cpu"

In [9]:
def schur_test(n, output_type='real'):
    matrix = np.random.random((n,n))
    schur = scipy.linalg.schur(matrix, output=output_type)
    return matrix, schur

In [26]:
M, (D, H) = schur_test(4, 'real')

In [32]:
H

array([[ 0.61073197, -0.52105164, -0.55848273,  0.20882694],
       [ 0.29806745, -0.46478215,  0.51417357, -0.65632224],
       [ 0.57491171,  0.70739618, -0.17109143, -0.37389156],
       [ 0.45567399,  0.10987896,  0.62805278,  0.62115821]])

In [33]:
Q, R = scipy.linalg.qr(M)

In [35]:
Q @ Q.T

array([[ 1.00000000e+00, -2.08220183e-18, -2.18317103e-16,
        -1.35941905e-16],
       [-2.08220183e-18,  1.00000000e+00, -2.52771808e-17,
         3.14784969e-17],
       [-2.18317103e-16, -2.52771808e-17,  1.00000000e+00,
        -4.72271062e-17],
       [-1.35941905e-16,  3.14784969e-17, -4.72271062e-17,
         1.00000000e+00]])

In [38]:
np.tri(3, k=1).T

array([[1., 1., 1.],
       [1., 1., 1.],
       [0., 1., 1.]])

tensor([[0.0549, 0.1571, 0.9173],
        [0.5380, 0.3532, 0.1866],
        [0.5534, 0.4202, 0.3882]])

In [ ]:
Q_base = torch.nn.Parameter(torch.rand((128,128)))
H_base = torch.nn.Parameter(0.5*torch.randn((128,128)))
Q, _ = torch.linalg.qr(Q_base)
H = torch.triu(H_base, -1)

In [90]:
K = Q @ H @ Q.T

In [91]:
K_matrix = K.detach().numpy()

In [92]:
eigenvalues, eigenvectors = np.linalg.eig(K_matrix)

In [93]:
np.abs(eigenvalues)

array([3.1442742 , 2.6611707 , 2.6611707 , 2.7431698 , 2.7431698 ,
       2.737169  , 2.670524  , 2.4881806 , 2.414794  , 2.414794  ,
       2.205238  , 2.205238  , 2.2654655 , 2.2654655 , 2.222312  ,
       2.0047212 , 2.0047212 , 2.0494947 , 2.0494947 , 2.3952007 ,
       2.3952007 , 1.9675816 , 1.9675816 , 2.3528504 , 2.289251  ,
       2.289251  , 2.0667315 , 2.0667315 , 1.9095497 , 1.9095497 ,
       1.9280558 , 1.9280558 , 2.179381  , 1.9672664 , 2.0887802 ,
       2.0887802 , 1.8230429 , 1.8230429 , 2.0117493 , 2.0117493 ,
       2.079539  , 2.079539  , 1.7152473 , 1.7152473 , 1.7819477 ,
       1.7819477 , 1.5681005 , 1.5681005 , 1.9254401 , 1.9254401 ,
       1.8024323 , 1.8024323 , 1.6064899 , 1.6064899 , 1.6040604 ,
       1.6040604 , 1.505285  , 1.505285  , 1.8608301 , 1.8608301 ,
       1.7616938 , 1.7616938 , 1.6389188 , 1.6389188 , 1.6555367 ,
       1.6555367 , 1.4838225 , 1.4838225 , 1.5532345 , 1.5532345 ,
       1.6352781 , 1.5131572 , 1.5131572 , 1.3530297 , 1.35302

In [8]:
print(matrix)
print(schur)

[[0.0907847  0.66119537]
 [0.63608846 0.88229277]]
(array([[-0.27319806+0.j, -0.0251069 +0.j],
       [ 0.        +0.j,  1.24627553+0.j]]), array([[-0.87603384+0.j,  0.48224963+0.j],
       [ 0.48224963+0.j,  0.87603384+0.j]]))


In [2]:
class KoopmanDataset(torch.utils.data.Dataset):
    def __init__(self, data, window_size=20):
        # data: (Total_T, 2, W, H)
        self.data = torch.FloatTensor(data).permute(0,3,1,2)
        self.window_size = window_size

    def __len__(self):
        return len(self.data) - self.window_size

    def __getitem__(self, idx):
        # Returns a sequence of length 'window_size'
        return self.data[idx : idx + self.window_size]

In [3]:
datafile = np.load(os.path.join(".", "data", "T1492_x1151_y1_z127_c2.npz"))
data = datafile['timeseries']
dataset = KoopmanDataset(data, window_size=16)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

In [4]:
model = ConvAutoencoderKoop(128)

In [10]:
for X_T in loader:
    B, T, C, H, W = X_T.shape
    X = X_T.view(B * T, C, H, W)
    
    Z, X_recon = model(X)
    
    Z_T = Z.view(B, T, -1)
    Z_T_null = Z_T[:, :-1, :]
    Z_T_shift = Z_T[:, 1:, :]
    Z_null = Z_T_null.reshape(B*(T-1), -1)
    Z_shift = Z_T_shift.reshape(B*(T-1), -1)
    K = torch.linalg.lstsq(Z_null, Z_shift).solution
    
    print(Z_T.shape)
    Z_0 = Z_T[:, 0, :]
    print(Z_0.shape)
    predictions = [Z_0]
    for m in range(1, T):
        # z_m = z_0 @ (K^m)
        Z_m = torch.matmul(predictions[-1], K)
        predictions.append(Z_m)
    Z_pred = torch.stack(predictions, dim=1).view(B*T, -1)
    print(Z_pred.shape)
    X_pred = model.decoder(Z_pred)
    print(X_pred.shape)
    print(X.shape)
    print(X_recon.shape)
    break
    #print(Z_null.shape, Z_shift.shape, X_recon.shape, K.shape)

torch.Size([8, 16, 128])
torch.Size([8, 128])
torch.Size([128, 128])
torch.Size([128, 2, 1151, 127])
torch.Size([128, 2, 1151, 127])
torch.Size([128, 2, 1151, 127])


In [ ]:
dim = 20

matrix = np.random.random((dim, dim))

In [ ]:
eigvalues, eigvectors = np.linalg.eig(matrix)

In [ ]:
eigvalues

In [ ]:
eigvalues

In [ ]:
eigvalues_torch = torch.tensor(eigvalues, device=DEVICE)

In [41]:
difference_matrix = torch.exp(-10*torch.abs(eigvalues_torch.unsqueeze(0) - eigvalues_torch.unsqueeze(1)))

In [ ]:
difference_matrix

In [ ]:
torch.triu(difference_matrix)

In [ ]:
triuindex = torch.triu_indices(20, 20, offset=1)

In [ ]:
triuindex.shape